# AgentsVille Trip Planner - Project Assignment

Implementation of AI system to help one plan a trip to the wonderful city of AgentsVille!

The AI system will demonstrate advanced LLM reasoning techniques including:

1. **Role-Based Prompting** - Act as a specialized travel planner
2. **Chain-of-Thought Reasoning** - Step-by-step planning of itineraries
3. **ReAct Prompting** - Thought → Action → Observation cycles
4. **Feedback Loops** - Self-evaluation using tools in the ReAct loop to find mistakes and improve plans

It's going to simulate external API calls to gather weather data and activities. Then, process this information to create personalized travel itineraries based on interests and other constraints.

## Project Overview

This project implements an AI-powered travel planning system that generates, evaluates, and refines personalized vacation itineraries based on traveler preferences, budget, weather conditions, and activity availability.

### Features

- **Structured Vacation Planning**
  - Collects and validates vacation details such as destination, travel dates, interests, and budget using a Pydantic model (`VacationInfo`).

- **Weather and Activity Integration**
  - Simulates external API calls to retrieve weather forecasts and available activities for each travel date, providing the information required to build realistic itineraries.

- **AI Itinerary Generation**
  - Uses an LLM-powered agent to create detailed day-by-day travel plans.
  - The agent is guided by a carefully designed prompt that incorporates traveler preferences, trip constraints, and contextual information to generate comprehensive itineraries in a single model call.

- **Automatic Itinerary Evaluation**
  - Validates the generated itinerary against multiple quality criteria, including:
    - Destination and travel date consistency.
    - Budget accuracy.
    - Availability of recommended activities.
    - Weather suitability for each planned activity.
  - Includes LLM-based evaluations for criteria that require semantic reasoning.

- **Tool-Based Agent Architecture**
  - The planning agent is supported by specialized tools that perform deterministic tasks, including:
    - `calculator_tool` for accurate cost calculations.
    - `get_activities_by_date_tool` for retrieving activities available on a specific date.
    - `run_evals_tool` for validating the generated itinerary.
    - `final_answer_tool` for returning the final itinerary in a structured format.

- **Iterative Itinerary Revision**
  - Implements a second AI agent that refines the initial itinerary using the ReAct (Thought → Action → Observation) reasoning framework.
  - The agent iteratively invokes tools, analyzes their outputs, and improves the itinerary until all evaluation criteria are satisfied.
  - Incorporates traveler feedback to ensure the final plan better reflects user preferences while satisfying constraints such as maintaining at least two activities per day.

- **Narrative Trip Summary**
  - Generates a concise and engaging narrative summarizing the trip, highlighting the most memorable activities and experiences included in the final itinerary.

## Initial Setup

In [ ]:
# Install required packages if not already installed
%pip install -q json-repair==0.47.1 numexpr==2.11.0 openai==1.74.0 pandas==2.3.0 pydantic==2.11.7 python-dotenv==1.1.0


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="****",
    api_key=os.getenv(
         "OPENAI_API_KEY"
    ),
)


In [ ]:
# You can experiment different OpenAI models.
# I will use GPT-4.1-mini, which is a good balance of speed and cost.
from enum import Enum

class OpenAIModel(str, Enum):
    GPT_41 = "gpt-4.1"  # Strong default choice for development tasks, particularly those requiring speed, responsiveness, and general-purpose reasoning. 
    GPT_41_MINI = "gpt-4.1-mini"  # Fast and affordable, good for brainstorming, drafting, and tasks that don't require the full power of GPT-4.1.
    GPT_41_NANO = "gpt-4.1-nano"  # The fastest and cheapest model, suitable for lightweight tasks, high-frequency usage, and edge computing.

MODEL = OpenAIModel.GPT_41_MINI  # Default model for this project


## Define Vacation Details

Let's encode the details of vacation in JSON format and verify it using Pydantic.

In practice, a chatbot agent could gather the information of a user. After it has gathered all the information it needs, it could generate this JSON object from the chat transcript. We will skip that step to focus on the itinerary generation itself.

In [ ]:
# The Vacation Info Data Structure

VACATION_INFO_DICT = {
    "travelers": [
        {
            "name": "Yuri",
            "age": 30,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, movies,
            # music, photography, reading, sports, technology, theatre, tennis, writing
            "interests": ["tennis", "cooking", "comedy", "technology"],
            "cuisine_preferences": ["Brazilian", "Thai", "Mexican"]

        },
        {
            "name": "Hiro",
            "age": 25,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, movies,
            # music, photography, reading, sports, technology, theatre, tennis, writing
            "interests": ["reading", "music", "theatre", "art"],
            "cuisine_preferences": ["Indian", "Japanese"]
        },
        {
            "name": "Anna",
            "age": 26,
            # Possible interests: art, cooking, comedy, dancing, fitness, gardening, hiking, movies,
            # music, photography, reading, sports, technology, theatre, tennis, writing
            "interests": ["photography", "sports", "dancing", "movies"],
            "cuisine_preferences": ["Italian", "Mediterranean", "Middle Eastern"]
        },
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",  # Mock data exists for 2025-06-10
    "date_of_departure": "2025-06-12",  # ...until 2025-06-15.
    "budget": 130,  # Budget is in fictional currency units.
}

In [ ]:
# Validate the data structure using Pydantic

from project_lib import Interest, Cuisine
from typing import List
from pydantic import BaseModel
import datetime
from pprint import pprint

class Traveler(BaseModel):
    """A traveler with a name, age, and list of interests.
    
    Attributes:
        name (str): The name of the traveler.
        age (int): The age of the traveler.
        interests (List[Interest]): A list of interests of the traveler.
    """
    name: str
    age: int
    interests: List[Interest]
    cuisine_preferences: List[Cuisine]

class VacationInfo(BaseModel):
    """Vacation information including travelers, destination, dates, and budget.
    Attributes:
        travelers (List[Traveler]): A list of travelers.
        destination (str): The vacation destination.
        date_of_arrival (datetime.date): The date of arrival.
        date_of_departure (datetime.date): The date of departure.
        budget (int): The budget for the vacation in fictional currency units.
    """
    
    travelers: List[Traveler]
    destination: str
    date_of_arrival: datetime.date
    date_of_departure: datetime.date
    budget: int


# Validate the VacationInfo data structure
vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)
# Display the vacation info as a dictionary
pprint(vacation_info.model_dump())

# Check that VacationInfo contains the expected data
assert "travelers" in vacation_info.model_dump().keys(), "VacationInfo should contain 'travelers' key"
assert "destination" in vacation_info.model_dump().keys(), "VacationInfo should contain 'destination' key"
assert "date_of_arrival" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_arrival' key"
assert "date_of_departure" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_departure' key"
assert "budget" in vacation_info.model_dump().keys(), "VacationInfo should contain 'budget' key"
assert isinstance(vacation_info.travelers, list), "Travelers should be a list"
assert all(isinstance(traveler, Traveler) for traveler in vacation_info.travelers), "All travelers should be instances of Traveler class"
assert isinstance(vacation_info.date_of_arrival, datetime.date), "date_of_arrival should be a date"
assert isinstance(vacation_info.date_of_departure, datetime.date), "date_of_departure should be a date"
assert isinstance(vacation_info.budget, int), "budget should be an integer"

# If all assertions pass, print a success message
print("✅ VacationInfo data structure is valid!")

{'budget': 130,
 'date_of_arrival': datetime.date(2025, 6, 10),
 'date_of_departure': datetime.date(2025, 6, 12),
 'destination': 'AgentsVille',
 'travelers': [{'age': 30,
                'cuisine_preferences': [Brazilian, Thai, Mexican],
                'interests': [tennis, cooking, comedy, technology],
                'name': 'Yuri'},
               {'age': 25,
                'cuisine_preferences': [Indian, Japanese],
                'interests': [reading, music, theatre, art],
                'name': 'Hiro'},
               {'age': 26,
                'cuisine_preferences': [Italian, Mediterranean, Middle Eastern],
                'interests': [photography, sports, dancing, movies],
                'name': 'Anna'}]}
✅ VacationInfo data structure is valid!


## Review Weather and Activity Schedules

Now that we have the trip details, we can retrieve the weather and activity schedules for the dates of the trip.

We will call an API to get all the data at once, in order to be able to include it in the context for the itinerary planning agent.

Also, we will format the data as Pandas DataFrames for easier viewing.

In [7]:
vacation_info.date_of_arrival

datetime.date(2025, 6, 10)

In [ ]:
# The `call_weather_api_mocked` mocks calling a weather API to get weather data

from project_lib import call_weather_api_mocked
import pandas as pd

pd.set_option("display.max_colwidth", None)  # Show full content in DataFrame cells

weather_for_dates = [
    call_weather_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
    for ts in pd.date_range(
        start=vacation_info.date_of_arrival,
        end=vacation_info.date_of_departure,
        freq="D",
    )
]

weather_for_dates_df = pd.DataFrame(weather_for_dates)

weather_for_dates_df

,date,city,temperature,temperature_unit,condition,description
0,2025-06-10,AgentsVille,31,celsius,clear,A bright and sunny day in AgentsVille with clear skies and warm temperatures. Perfect weather for outdoor activities!
1,2025-06-11,AgentsVille,34,celsius,partly cloudy,"A warm day with periods of sunshine and mixed clouds, making it a perfect opportunity for outdoor activities."
2,2025-06-12,AgentsVille,28,celsius,thunderstorm,"A thunderstorm is expected to roll in during the afternoon, bringing heavy rain and gusty winds. The atmosphere will feel charged with humidity, creating a sultry and dramatic setting as clouds build in the sky."


In [11]:
vacation_info.model_dump()

{'travelers': [{'name': 'Yuri',
   'age': 30,
   'interests': [tennis, cooking, comedy, technology],
   'cuisine_preferences': [Brazilian, Thai, Mexican]},
  {'name': 'Hiro',
   'age': 25,
   'interests': [reading, music, theatre, art],
   'cuisine_preferences': [Indian, Japanese]},
  {'name': 'Anna',
   'age': 26,
   'interests': [photography, sports, dancing, movies],
   'cuisine_preferences': [Italian, Mediterranean, Middle Eastern]}],
 'destination': 'AgentsVille',
 'date_of_arrival': datetime.date(2025, 6, 10),
 'date_of_departure': datetime.date(2025, 6, 12),
 'budget': 130}

In [ ]:
# The `call_activities_api_mocked` function returns the activities for a given date and city.

from project_lib import call_activities_api_mocked

activities_for_dates = [
    activity
    for ts in pd.date_range(
        start=vacation_info.date_of_arrival,
        end=vacation_info.date_of_departure,
        freq="D",
    )
    for activity in call_activities_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
]

activities_for_dates_df = pd.DataFrame(activities_for_dates)

activities_for_dates_df

,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00,2025-06-10 11:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00,2025-06-10 13:30,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"
2,event-2025-06-10-2,Artful Athletics: Paint & Play Extravaganza,2025-06-10 15:00,2025-06-10 17:00,"Creative Courts Park, AgentsVille","Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide! At 'Artful Athletics: Paint & Play Extravaganza', you'll participate in collaborative outdoor murals inspired by your favorite sports, and then get moving with fun, friendly sports mini-games. Whether you love painting or playing, this event celebrates creativity, teamwork, and the joy of movement under the open sky. Perfect for art lovers and sports enthusiasts alike—come ready to express yourself and get active! (Event is held outdoors; in case of rain, we move indoors to the Community Gym nearby.)",15,"[art, sports]"
3,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"
4,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00,2025-06-11 10:30,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
5,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00,2025-06-11 13:30,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI cr

## The ItineraryAgent

First we will review the Pydantic objects used for defining the output of our agent, the TravelPlan, ItineraryDay, Activity, and Weather classes.

Second, we will create a Chain-of-Thought prompt to guide the agent in planning the trip. This prompt will instruct the agent to consider the weather, activities, and user preferences when creating the itinerary.

Third, we will run the agent to produce the TravelPlan object, which will will refine in the following steps.

In [14]:
vacation_info.model_dump()

{'travelers': [{'name': 'Yuri',
   'age': 30,
   'interests': [tennis, cooking, comedy, technology],
   'cuisine_preferences': [Brazilian, Thai, Mexican]},
  {'name': 'Hiro',
   'age': 25,
   'interests': [reading, music, theatre, art],
   'cuisine_preferences': [Indian, Japanese]},
  {'name': 'Anna',
   'age': 26,
   'interests': [photography, sports, dancing, movies],
   'cuisine_preferences': [Italian, Mediterranean, Middle Eastern]}],
 'destination': 'AgentsVille',
 'date_of_arrival': datetime.date(2025, 6, 10),
 'date_of_departure': datetime.date(2025, 6, 12),
 'budget': 130}

In [ ]:
# Review the data structure we will use for representing a TravelPlan, which includes
# weather, activity_recommendations, and itinerary for each day of the trip.

class Weather(BaseModel):
    temperature: float
    temperature_unit: str
    condition: str


class Activity(BaseModel):
    activity_id: str
    name: str
    start_time: str
    end_time: str
    location: str
    description: str
    price: int
    related_interests: List[Interest]


class ActivityRecommendation(BaseModel):
    activity: Activity
    reasons_for_recommendation: List[str]


class ItineraryDay(BaseModel):
    date: datetime.date
    weather: Weather
    activity_recommendations: List[ActivityRecommendation]


class TravelPlan(BaseModel):
    city: str
    start_date: datetime.date
    end_date: datetime.date
    total_cost: int
    itinerary_days: List[ItineraryDay]

In [45]:
activities_for_dates_df

,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00,2025-06-10 11:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00,2025-06-10 13:30,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"
2,event-2025-06-10-2,Artful Athletics: Paint & Play Extravaganza,2025-06-10 15:00,2025-06-10 17:00,"Creative Courts Park, AgentsVille","Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide! At 'Artful Athletics: Paint & Play Extravaganza', you'll participate in collaborative outdoor murals inspired by your favorite sports, and then get moving with fun, friendly sports mini-games. Whether you love painting or playing, this event celebrates creativity, teamwork, and the joy of movement under the open sky. Perfect for art lovers and sports enthusiasts alike—come ready to express yourself and get active! (Event is held outdoors; in case of rain, we move indoors to the Community Gym nearby.)",15,"[art, sports]"
3,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"
4,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00,2025-06-11 10:30,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
5,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00,2025-06-11 13:30,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI cr

In [ ]:
# Chain-of-Thought (CoT) prompt for the Itinerary Agent.
# Components of the prompt:
# [Role]: e.g. Itinerary Planning Agent
# [Task]: Explicitly or implicitly define a step-by-step process for creating the itinerary.
# [Output Format]: Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the specified format.
# [Examples, optional]: Provide examples of how the output should look like.
# [Context]: Make sure to include the weather and activities data in the context, otherwise the agent won't have access to it
# and may instead hallucinate the data.

import json 
from project_lib import ChatAgent
from typing import Optional

ITINERARY_AGENT_SYSTEM_PROMPT = f"""
You are an Itinerary Planning Agent that generates a detailed, day-by-day travel itinerary for AgentsVille.

TASK

Your taks is to consider the user profile availabe in {vacation_info} to generate a personalized 
and detailed travel itinerary, so that the person makes the most of their trip within their budget.
You should also avoid outdoor-only activities. 
In order to generate the itinerary, you must consider the weather and, in case of rain, 
events should be chosen based on traveler's interests.
Note that, the person's budget should not be exceeded 
and there should be at least one activity per day.
You should also ensure that the suggested activity_id exist inside the {activities_for_dates_df};

OUTPUT FORMAT

Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:

    ANALYSIS:
    Give the user a catchy phrase and a name for the itinerary. Then, display the response according to the following:
    Adventure dates: start and end dates;
    Selected activities, day by day, and the reasons for selection, based on the user's interests, budget, weather, etc.
    How much will it cost?: just a number or a range. Example: 100 - 130.

    FINAL OUTPUT:

    ```json
    
{{
        "city": "string",
        "start_date": "date(YYYY-MM-DD)",
        "end_date": "date(YYYY-MM-DD)",
        "total_cost": "int",
        "itinerary_days": [
          {{
            "date": "date(YYYY-MM-DD)",
            "weather": {{
              "temperature": "float",
              "temperature_unit": "string",
              "condition": "string"
            }},
            "activity_recommendations": [
              {{
                "activity": {{
                  "activity_id": "string",
                  "name": "string",
                  "start_time": "string (YYYY-MM-DD HH:MM)",
                  "end_time": "string (YYYY-MM-DD HH:MM)",
                  "location": "string",
                  "description": "string",
                  "price": "int",
                  "related_interests": [
                    "Interest"
                  ]
                }},
                "reasons_for_recommendation": [
                  "string"
                ]
              }}
            ]
          }}
        ]
    }}


    ```

## Context

Check the weather in {weather_for_dates_df};
Check the activities in {activities_for_dates_df};
Follow the following models to structure your final json output:
{TravelPlan};
{ItineraryDay};
{Weather};
{Activity};
{ActivityRecommendation};
"""  # noqa


assert "TASK" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'TASK' section"
assert "OUTPUT FORMAT" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'OUTPUT FORMAT' section"


class ItineraryAgent(ChatAgent):
    """An agent that plans itineraries based on vacation information, weather, and activities."""
    system_prompt = ITINERARY_AGENT_SYSTEM_PROMPT

    def get_itinerary(self, vacation_info: VacationInfo, model: Optional[OpenAIModel] = None) -> TravelPlan:
        """Generates a travel itinerary based on the provided vacation information."""
        from project_lib import print_in_box
        response = (self.chat(
            user_message=vacation_info.model_dump_json(indent=2),
            add_to_messages=False,
            model=model or self.model,
        ) or "").strip()

        print_in_box(response, "Raw Response")

        # Parse the response
        json_text = response.strip()

        if "```json" in json_text:
            json_text = json_text.split("```json")[1].split("```")[0].strip()

        try:
            print("Validating json_text:")
            print(json_text)
            travel_plan = TravelPlan.model_validate_json(json_text)
            return travel_plan
        except Exception as e:
            print("Error validating the following text as TravelPlan JSON:")
            print(json_text)
            raise

itinerary_agent = ItineraryAgent(client=client, model=MODEL)


╔══════════════════════════════════════════[ ItineraryAgent - System Prompt ]══════════════════════════════════════════╗
║ You are an Itinerary Planning Agent that generates a detailed, day-by-day travel itinerary for AgentsVille.          ║
║ TASK                                                                                                                 ║
║ Your taks is to consider the user profile availabe in travelers=[Traveler(name='Yuri', age=30, interests=[tennis,    ║
║ cooking, comedy, technology], cuisine_preferences=[Brazilian, Thai, Mexican]), Traveler(name='Hiro', age=25,         ║
║ interests=[reading, music, theatre, art], cuisine_preferences=[Indian, Japanese]), Traveler(name='Anna', age=26,     ║
║ interests=[photography, sports, dancing, movies], cuisine_preferences=[Italian, Mediterranean, Middle Eastern])]     ║
║ destination='AgentsVille' date_of_arrival=datetime.date(2025, 6, 10) date_of_departure=datetime.date(2025, 6, 12)    ║
║ budget=130 to generate a pers

In [ ]:
# Generate the travel itinerary

travel_plan_1 = itinerary_agent.get_itinerary(
    vacation_info=vacation_info,
    model=MODEL,  
)

if travel_plan_1 is not None:
    print("✅ Initial itinerary generated successfully. Congratulations!")


╔═══════════════════════════════════════════[ ItineraryAgent - User Prompt ]═══════════════════════════════════════════╗
║ {                                                                                                                    ║
║   "travelers": [                                                                                                     ║
║     {                                                                                                                ║
║       "name": "Yuri",                                                                                                ║
║       "age": 30,                                                                                                     ║
║       "interests": [                                                                                                 ║
║         "tennis",                                                                                                    ║
║         "cooking",           

## Evaluating the Itinerary

### Itinerary Evaluation and Refinement

The project includes an evaluation layer to assess the quality and consistency of the generated itineraries.

- **Itinerary Evaluation**
  - Evaluates the generated plan against a set of predefined quality criteria.
  - Checks whether the itinerary satisfies the traveler's requirements and constraints.
  - Identifies issues that may require correction, such as invalid activities, incorrect calculations, or inconsistencies with the requested trip details.

- **Iterative Improvement**
  - The evaluation results are used as feedback for the itinerary revision agent.
  - Instead of relying solely on the initial LLM output, the system allows the agent to analyze its own plan and make improvements in a second pass.
  - This feedback loop enables the itinerary to be progressively refined until the required criteria are satisfied.

In [ ]:
# Helper functions for running the evaluation functions

class AgentError(Exception):
    pass


class EvaluationResults(BaseModel):
    success: bool
    failures: List[str]
    eval_functions: List[str]


def get_eval_results(vacation_info, final_output, eval_functions) -> EvaluationResults:
    """
    Evaluates the final output of the itinerary agent against a set of evaluation functions.
    Args:
        vacation_info (VacationInfo): The vacation information used to generate the itinerary.
        final_output (TravelPlan): The final output from the itinerary agent.
        eval_functions (List[callable]): A list of evaluation functions to apply.
    Returns:
        EvaluationResults: An object containing the success status, any failures, and the names of the evaluation functions used.
    """
    from project_lib import print_in_box
    if not isinstance(vacation_info, VacationInfo):
        raise ValueError("vacation_info must be an instance of VacationInfo")
    if not isinstance(final_output, TravelPlan):
        raise ValueError("final_output must be an instance of TravelPlan")
    if not isinstance(eval_functions, list) or not all(
        callable(fn) for fn in eval_functions
    ):
        raise ValueError("eval_functions must be a list of callable functions")
    eval_results = []
    for eval_fn in eval_functions:
        try:
            eval_fn(vacation_info, final_output)
        except AgentError as e:
            error_msg = str(e)
            print_in_box(error_msg, title="Evaluation Error")
            print("\n\n")

            eval_results.append(error_msg)
    return EvaluationResults(
        success=len(eval_results) == 0,
        failures=eval_results,
        eval_functions=[fn.__name__ for fn in eval_functions],
    )


In [ ]:
# Basic evaluation functions

def eval_start_end_dates_match(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the arrival and departure dates in vacation_info match the start and end dates in final_output.

    Args:
        vacation_info (dict): Contains the vacation details including arrival and departure dates
        final_output (dict): Contains the itinerary details including start and end dates

    Raises:
        AgentError: If either the arrival date doesn't match the start date or the departure date doesn't match the end date
    """
    if (
        vacation_info.date_of_arrival != final_output.start_date
        or vacation_info.date_of_departure != final_output.end_date
    ):
        raise AgentError(
            f"Dates do not match: {vacation_info.date_of_arrival} != {final_output.start_date} or {vacation_info.date_of_departure} != {final_output.end_date}"
        )

    if final_output.start_date > final_output.end_date:
        raise AgentError(
            f"Start date is after end date: {final_output.start_date} > {final_output.end_date}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_start_end_dates_match],
)

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match'])

In [ ]:
# Evaluation functions related to the budget and total cost

def eval_total_cost_is_accurate(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output matches the sum of all activity prices.

    Args:
        vacation_info (dict): Contains the vacation details
        final_output (dict): Contains the itinerary details including activities with prices and total cost

    Raises:
        AgentError: If the calculated total cost doesn't match the stated total cost
    """
    actual_total_cost = 0

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            actual_total_cost += activity_recommendation.activity.price

    stated_total_cost = int(final_output.total_cost)

    if actual_total_cost != stated_total_cost:
        raise AgentError(
            f"Stated total cost does not match calculated total cost: {actual_total_cost} != {stated_total_cost}"
        )
    
def eval_total_cost_is_within_budget(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verifies that the total cost stated in final_output is within the budget specified in vacation_info.

    Args:
        vacation_info (dict): Contains the vacation details including budget
        final_output (dict): Contains the itinerary details including total cost

    Raises:
        AgentError: If the total cost exceeds the budget
    """
    stated_total_cost = int(final_output.total_cost)
    if stated_total_cost > vacation_info.budget:
        raise AgentError(
            f"Total cost exceeds budget: {stated_total_cost} > {vacation_info.budget}"
        )

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_total_cost_is_accurate, eval_total_cost_is_within_budget],
)


EvaluationResults(success=True, failures=[], eval_functions=['eval_total_cost_is_accurate', 'eval_total_cost_is_within_budget'])

In [52]:
from project_lib import call_activity_by_id_api_mocked

In [ ]:
# The final output contains copies of the activities, so we need to verify that the copies are accurate
# and not hallucinated.

def eval_itinerary_events_match_actual_events(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the events listed in the itinerary match the actual events

    Args:
        vacation_info (dict): Contains the vacation details including traveler information and their interests
        final_output (dict): Contains the itinerary details including daily activities

    Raises:
        AgentError: If any traveler has no matching activities or if one traveler has more than twice
                   the number of matching activities compared to another traveler
    """
    from project_lib import call_activity_by_id_api_mocked
    event_ids_not_matching = []
    event_ids_missing = []

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            event_id = activity_recommendation.activity.activity_id
            # Assuming get_event_by_id is a function that retrieves the event by its ID

            reference_event = call_activity_by_id_api_mocked(event_id)

            if reference_event is None:
                event_ids_missing.append(event_id)

            elif Activity(**reference_event) != activity_recommendation.activity:
                
                print(
                    "---\n"
                    f"Event ID {event_id} does not match the reference event:\n"
                    f"Reference Event: {reference_event}\n"
                    f"Activity Event: {activity_recommendation.activity.model_dump()}"
                )
                event_ids_not_matching.append(event_id)
            else:
                # The event matches, so we can continue
                pass

    if event_ids_missing or event_ids_not_matching:
        raise AgentError(
            f"Event IDs missing: {event_ids_missing}\nEvent IDs not matching: {event_ids_not_matching}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_events_match_actual_events],
)


EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_events_match_actual_events'])

In [ ]:
# Check that the itinerary includes at least one activity matching each traveler's interests.

def eval_itinerary_satisfies_interests(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that the itinerary includes activities matching each traveler's interests.

    This function checks that each traveler has at least one activity in the itinerary that matches their interests.

        Args:
        vacation_info (dict): Contains the vacation details including traveler information and their interests
        final_output (dict): Contains the itinerary details including daily activities

    Raises:
        AgentError: If any traveler has no matching activities or if one traveler has more than twice
                   the number of matching activities compared to another traveler
    """
    traveler_to_interests = {}
    traveler_to_interest_hit_counts = {}

    for traveler in vacation_info.travelers:
        traveler_to_interests[traveler.name] = traveler.interests
        traveler_to_interest_hit_counts[traveler.name] = 0

    for traveler_name, interests in traveler_to_interests.items():
        for itinerary_day in final_output.itinerary_days:
            for activity_recommendation in itinerary_day.activity_recommendations:
                # Check if the activity matches any of the traveler's interests
                matching_interests = set(traveler_to_interests[traveler_name]) & set(
                    activity_recommendation.activity.related_interests
                )

                if matching_interests:
                    traveler_to_interest_hit_counts[traveler_name] += 1
                    print(
                        f"✅ Traveler {traveler_name} has a match with interest {matching_interests} at {activity_recommendation.activity.name}"
                    )

    travelers_with_no_interest_hits = [
        traveler
        for traveler, interest_hit_count in traveler_to_interest_hit_counts.items()
        if interest_hit_count == 0
    ]

    # If any of the travelers have 0 matches, raise an error
    if travelers_with_no_interest_hits:
        raise AgentError(
            f"Travelers {travelers_with_no_interest_hits} has no matches with the itinerary."
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_satisfies_interests],
)


✅ Traveler Yuri has a match with interest {technology} at FutureTech Breakfast Meet-Up
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art, reading} at AgentsVille Twilight Writing Escape
✅ Traveler Hiro has a match with interest {music} at Morning Groove Dance Party
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Anna has a match with interest {dancing} at Morning Groove Dance Party
✅ Traveler Anna has a match with interest {photography} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Anna has a match with interest {movies} at Tech & Film Fusion Night


EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_satisfies_interests'])

In [18]:
weather_for_dates_df

,date,city,temperature,temperature_unit,condition,description
0,2025-06-10,AgentsVille,31,celsius,clear,A bright and sunny day in AgentsVille with clear skies and warm temperatures. Perfect weather for outdoor activities!
1,2025-06-11,AgentsVille,34,celsius,partly cloudy,"A warm day with periods of sunshine and mixed clouds, making it a perfect opportunity for outdoor activities."
2,2025-06-12,AgentsVille,28,celsius,thunderstorm,"A thunderstorm is expected to roll in during the afternoon, bringing heavy rain and gusty winds. The atmosphere will feel charged with humidity, creating a sultry and dramatic setting as clouds build in the sky."


In [17]:
activities_for_dates_df

,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00,2025-06-10 11:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00,2025-06-10 13:30,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"
2,event-2025-06-10-2,Artful Athletics: Paint & Play Extravaganza,2025-06-10 15:00,2025-06-10 17:00,"Creative Courts Park, AgentsVille","Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide! At 'Artful Athletics: Paint & Play Extravaganza', you'll participate in collaborative outdoor murals inspired by your favorite sports, and then get moving with fun, friendly sports mini-games. Whether you love painting or playing, this event celebrates creativity, teamwork, and the joy of movement under the open sky. Perfect for art lovers and sports enthusiasts alike—come ready to express yourself and get active! (Event is held outdoors; in case of rain, we move indoors to the Community Gym nearby.)",15,"[art, sports]"
3,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"
4,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00,2025-06-11 10:30,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
5,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00,2025-06-11 13:30,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI cr

In [ ]:
# Using LLM to determine whether an event should be avoided due to weather conditions.


ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
You will help to determine whether an event should be avoided due to weather conditions


Task
If activities are carried out outdoors, according to their description and location, avoid if the 
weather is thunderstorm.
When there is not enough information, try to infer based on the description, or assume the activity IS_COMPATIBLE.
Apart from that, depending on the weather condition, also look outfor backup options mentioned in the activity description.

Output format

    REASONING:
    Activity: activity's name;
    Classification: IS_COMPATIBLE or IS_INCOMPATIBLE
    Reason: describe the reason why you came with the classification. 

    FINAL ANSWER:
    [IS_COMPATIBLE, IS_INCOMPATIBLE]

Examples
`
Activity id: event-2025-06-10-1;
Activity name: Serve & Savor: Tennis and Taste Luncheon;
Activiry description: Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, 
then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. 
Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!
Weather condition: thunderstorm;
Classification: IS_INCOMPATIBLE

Activity id: event-2025-06-10-3;
Activity name: AgentsVille Twilight Writing Escape;
Activiry description: Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. 
Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven
Weather condition: thunderstorm;
Classification: IS_COMPATIBLE

""".strip()



def eval_activities_and_weather_are_compatible(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that no outdoor-only activities are scheduled during inclement weather conditions.

    Args:
        vacation_info (dict): Contains the vacation details
        final_output (dict): Contains the itinerary details including daily activities and weather conditions

    Raises:
        AgentError: If any outdoor activities are scheduled during weather conditions that could ruin them
    """
    from project_lib import do_chat_completion

    activities_that_are_incompatible = []

    for itinerary_day in final_output.itinerary_days:
        weather_condition = itinerary_day.weather.condition

        for activity_recommendation in itinerary_day.activity_recommendations:
            resp = do_chat_completion(
                messages=[
                    {
                        "role": "system",
                        "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": f"Activity: {activity_recommendation.activity.name}\nDescription: {activity_recommendation.activity.description}\nWeather Condition: {weather_condition}",
                    },
                ],
                client=client,
                # This is a high-frequency use case, so we use a fast and cheap model.
                model=OpenAIModel.GPT_41_NANO,
            )
    


            if "IS_COMPATIBLE" in (resp or ""):
                is_compatible = True
            elif "IS_INCOMPATIBLE" in (resp or ""):
                is_compatible = False
            else:
                raise RuntimeError(
                    f"Unexpected response from the model: {resp}. Expected 'IS_COMPATIBLE' or 'IS_INCOMPATIBLE'."
                )

            if is_compatible:
                print(
                    f"✅ Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are compatible."
                )

            else:
                activities_that_are_incompatible.append(
                    activity_recommendation.activity.name
                )
                print(
                    f"❌ Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are incompatible."
                )

    if activities_that_are_incompatible:
        raise AgentError(
            f"Activities that may be ruined by inclement weather: {activities_that_are_incompatible}"
        )


eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[
        eval_activities_and_weather_are_compatible
    ],
)

eval_results

✅ Activity FutureTech Breakfast Meet-Up (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity AgentsVille Twilight Writing Escape (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Morning Groove Dance Party (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Tech Lunch & Learn: AI Frontiers (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Palette & Palate: Art Meets Cooking Experience (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.


EvaluationResults(success=True, failures=[], eval_functions=['eval_activities_and_weather_are_compatible'])

In [ ]:
# Run all of the evaluation functions again

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
]

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

eval_results.model_dump()

✅ Traveler Yuri has a match with interest {technology} at FutureTech Breakfast Meet-Up
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art, reading} at AgentsVille Twilight Writing Escape
✅ Traveler Hiro has a match with interest {music} at Morning Groove Dance Party
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Anna has a match with interest {dancing} at Morning Groove Dance Party
✅ Traveler Anna has a match with interest {photography} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Anna has a match with interest {movies} at Tech & Film Fusion Night
✅ Activity FutureTech Br

{'success': True,
 'failures': [],
 'eval_functions': ['eval_start_end_dates_match',
  'eval_total_cost_is_accurate',
  'eval_itinerary_events_match_actual_events',
  'eval_itinerary_satisfies_interests',
  'eval_total_cost_is_within_budget',
  'eval_activities_and_weather_are_compatible']}

## Defining the Tools

The ItineraryRevisionAgent will be a ReAct-based agent that will use tools to:
- Evaluate/Re-evaluate the itinerary
- Use a calculator since LLMs sometimes struggle with arithmetic
- Call the activities API to get more information about activities
- Return the final itinerary


In [ ]:
# Helper function to generate tool descriptions from function docstrings

def get_tool_descriptions_string(fns):
    """Generates a tool description from a function's docstring.
    Args:
        fns (list): List of functions to generate descriptions for.
    Returns:
        str: A formatted string containing the function names and their descriptions."""
    resp = ""
    for fn in fns:
        function_name = fn.__name__
        function_doc = fn.__doc__ or "No description provided."

        resp += f"* `{function_name}`: {function_doc}\n"

    return resp

In [ ]:
# Define the calculator tool that evaluates mathematical expressions.

def calculator_tool(input_expression) -> float:
    """Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    """
    import numexpr as ne
    return float(ne.evaluate(input_expression))


assert calculator_tool("1 + 1") == 2.0

print(get_tool_descriptions_string([calculator_tool]))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

Args:
    input_expression (str): A string containing a valid mathematical expression to evaluate.

Returns:
    float: The result of the evaluated expression.

Example:
    >>> calculator_tool("1 + 1")
    2.0




In [ ]:
# Tool to fetch activities for a given date and city.

def get_activities_by_date_tool(date: str, city: str) -> List[dict]:
    """
        Fetch activities for a given date and city

        Args:
            date (str): A string containing a valid date format (YYYY/MM/DD)
            city (str): A string containing the city of interest.
        
        Returns:
            List: A list containing dictionaries that describe activities.

        Example:
            >> get_activities_by_date_tool("2025-06-10", "AgentsVille")
            [{'activity_id': 'event-2025-06-10-0',
            'name': 'FutureTech Breakfast Meet-Up',
            'start_time': datetime.datetime(2025, 6, 10, 9, 0),
            'end_time': datetime.datetime(2025, 6, 10, 11, 0),
            'location': 'The Innovation Atrium, Tech District, AgentsVille',
            'description': '...',
            'price': 20,
            'related_interests': [technology]},
            {'activity_id': 'event-2025-06-10-1',
            'name': 'Serve & Savor: Tennis and Taste Luncheon',
            'start_time': datetime.datetime(2025, 6, 10, 12, 0),
            'end_time': datetime.datetime(2025, 6, 10, 13, 30),
            'location': 'The Grand Racquet Terrace, AgentsVille',
            'description': "...",
            'price': 20,
            'related_interests': [cooking, tennis]},


    """
    from project_lib import call_activities_api_mocked
    resp = call_activities_api_mocked(date=date, city=city)

    return [Activity.model_validate(activity).model_dump() for activity in resp]



assert len(get_activities_by_date_tool("2025-06-10", "AgentsVille")) > 0

print(get_tool_descriptions_string([get_activities_by_date_tool]))
#get_activities_by_date_tool("2025-06-10", "AgentsVille")

* `get_activities_by_date_tool`: 
Fetch activities for a given date and city

Args:
    date (str): A string containing a valid date format (YYYY/MM/DD)
    city (str): A string containing the city of interest.

Returns:
    List: A list containing dictionaries that describe activities.

Example:
    >> get_activities_by_date_tool("2025-06-10", "AgentsVille")
    [{'activity_id': 'event-2025-06-10-0',
    'name': 'FutureTech Breakfast Meet-Up',
    'start_time': datetime.datetime(2025, 6, 10, 9, 0),
    'end_time': datetime.datetime(2025, 6, 10, 11, 0),
    'location': 'The Innovation Atrium, Tech District, AgentsVille',
    'description': '...',
    'price': 20,
    'related_interests': [technology]},
    {'activity_id': 'event-2025-06-10-1',
    'name': 'Serve & Savor: Tennis and Taste Luncheon',
    'start_time': datetime.datetime(2025, 6, 10, 12, 0),
    'end_time': datetime.datetime(2025, 6, 10, 13, 30),
    'location': 'The Grand Racquet Terrace, AgentsVille',
    'description': 

In [ ]:
# Tool to run all evaluation functions on a travel plan.


def run_evals_tool(travel_plan: TravelPlan) -> dict:
    """Runs all evaluation tools on the provided travel plan and vacation info.

    Args:
        travel_plan (TravelPlan): The travel plan to evaluate.

    Returns:
        EvaluationResults: The results of the evaluations.
    """
    if isinstance(travel_plan, dict):
        travel_plan = TravelPlan.model_validate(travel_plan)

    resp = get_eval_results(
        vacation_info=vacation_info,
        final_output=travel_plan,
        eval_functions=ALL_EVAL_FUNCTIONS,
    )
    return {
        # Show the success status and any failures
        "success": resp.success,
        "failures": resp.failures,
    }


print(get_tool_descriptions_string([run_evals_tool]))

* `run_evals_tool`: Runs all evaluation tools on the provided travel plan and vacation info.

Args:
    travel_plan (TravelPlan): The travel plan to evaluate.

Returns:
    EvaluationResults: The results of the evaluations.




In [ ]:
# Double checking that the tool works as expected.
# Should see the same results as before
run_evals_tool(travel_plan=travel_plan_1)

✅ Traveler Yuri has a match with interest {technology} at FutureTech Breakfast Meet-Up
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art, reading} at AgentsVille Twilight Writing Escape
✅ Traveler Hiro has a match with interest {music} at Morning Groove Dance Party
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Anna has a match with interest {dancing} at Morning Groove Dance Party
✅ Traveler Anna has a match with interest {photography} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Anna has a match with interest {movies} at Tech & Film Fusion Night
✅ Activity FutureTech Br

{'success': True, 'failures': []}

In [ ]:
# A tool to return the final travel plan

def final_answer_tool(final_output: TravelPlan) -> TravelPlan:
    """Returns the final travel plan

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    """
    return final_output


print(get_tool_descriptions_string([final_answer_tool]))

* `final_answer_tool`: Returns the final travel plan

Args:
    final_output (TravelPlan): The final travel plan to return.

Returns:
    TravelPlan: The final travel plan.




In [ ]:
# List of all tools available for the agent

ALL_TOOLS = [
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
]
print(get_tool_descriptions_string(ALL_TOOLS))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

Args:
    input_expression (str): A string containing a valid mathematical expression to evaluate.

Returns:
    float: The result of the evaluated expression.

Example:
    >>> calculator_tool("1 + 1")
    2.0

* `get_activities_by_date_tool`: 
Fetch activities for a given date and city

Args:
    date (str): A string containing a valid date format (YYYY/MM/DD)
    city (str): A string containing the city of interest.

Returns:
    List: A list containing dictionaries that describe activities.

Example:
    >> get_activities_by_date_tool("2025-06-10", "AgentsVille")
    [{'activity_id': 'event-2025-06-10-0',
    'name': 'FutureTech Breakfast Meet-Up',
    'start_time': datetime.datetime(2025, 6, 10, 9, 0),
    'end_time': datetime.datetime(2025, 6, 10, 11, 0),
    'location': 'The Innovation Atrium, Tech District, AgentsVille',
    'description': '...',
    'price': 20,
    'related_interests'

## The ItineraryRevisionAgent

The ItineraryRevisionAgent will
* take initial feedback from the user about the itinerary and
* use the tools defined above

to refine original itinerary iteratively using a ReAct-based approach.

In [ ]:
# Get the traveler's feedback and create a new evaluation function to check if the feedback was incorporated.

TRAVELER_FEEDBACK = "I want to have at least two activities per day."


def eval_traveler_feedback_is_incorporated(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Checks if the traveler's feedback was incorporated into the revised travel plan.

    Args:
        vacation_info (VacationInfo): The vacation information.
        final_output (TravelPlan): The revised travel plan.

    Raises:
        AgentError: If the traveler's feedback was not successfully incorporated.
    """

    agent = ChatAgent(
        system_prompt="""You are an expert in evaluating whether a travel plan incorporates traveler feedback.

    ## Output Format

    Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:

        ANALYSIS:
        * [step-by-step analysis]


        FINAL OUTPUT:
        [FULLY_INCORPORATED, PARTIALLY_INCORPORATED, NOT_INCORPORATED, or UNKNOWN]
        REASON: [reasoning for the final output]

    """,
        client=client,
        model=OpenAIModel.GPT_41,  # Use a powerful model for checking traveler feedback
    )

    resp = agent.chat(
        f"""Traveler Feedback: {TRAVELER_FEEDBACK}
    Revised Travel Plan: {final_output.model_dump_json()}
    """,
    )
    if "FINAL OUTPUT:" not in resp:
        raise RuntimeError(
            f"Unexpected response from the model: {resp}. Expected 'FINAL OUTPUT:'."
        )
    if "FULLY_INCORPORATED" not in resp:
        final_output = resp.split("FINAL OUTPUT:")[-1].strip()
        raise AgentError(
            f"Traveler feedback was not successfully incorporated into the revised travel plan. Response: {final_output}"
        )

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
    eval_traveler_feedback_is_incorporated, 
]

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

✅ Traveler Yuri has a match with interest {technology} at FutureTech Breakfast Meet-Up
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art, reading} at AgentsVille Twilight Writing Escape
✅ Traveler Hiro has a match with interest {music} at Morning Groove Dance Party
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Anna has a match with interest {dancing} at Morning Groove Dance Party
✅ Traveler Anna has a match with interest {photography} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Anna has a match with interest {movies} at Tech & Film Fusion Night
✅ Activity FutureTech Br

EvaluationResults(success=False, failures=['Traveler feedback was not successfully incorporated into the revised travel plan. Response: PARTIALLY_INCORPORATED\nREASON: The revised plan includes at least two activities on the first two days, but only provides one activity on the last day, which does not fully meet the traveler’s feedback of having at least two activities per day.'], eval_functions=['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [ ]:
# Define the ReAct system prompt for the Itinerary Revision Agent.

from project_lib import print_in_box

ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
Your job is to analyze the current TravelPlan and decide the next tool to call 
(without executing it), along with the correct arguments, until the plan is ready.

## Task

{TRAVELER_FEEDBACK}
1. Verify whether the TravelPlan satisfies all TRAVELER_FEEDBACK.
2. Ensure every activity in the TravelPlan has a matching ID in activities_for_dates_df.
3. If corrections are needed, refine the TravelPlan accordingly. 
4. The plan **must have** at least two activities per day.
5. If activities are carried out outdoors, according to their description and location, avoid them if the weather is thunderstorm.
6. Ensure all new suggested activities are compatible with the weather (use an evaluation tool to confirm this).
7. Confirm that the total trip cost fits within the traveler's budget by preparing the correct calculator expression whenever a calculation is required.
8. Once you have a valid TravelPlan, decide to use run_evals_tool exactly once using the TravelPlan as the argument.
9. After evaluating, decide to use final_answer_tool to return the final TravelPlan.


## Available Tools

Within ALL_TOOLS, you'll find the following tools you can use 
(you must only decide which one is next and with what arguments; do not execute them):


- calculator_tool
  - Purpose: arithmetic calculations for totals, differences, and budget checks.
  - Arguments: input_expression: <string>
  - ACTION packaging rule (for this policy's Output Format): put the expression under the key "input_expression" inside "arguments".
  - Example ACTION:
    {{"tool_name":"calculator_tool","arguments":{{"input_expression":"1200 + 3"}}}}
    {{"tool_name":"calculator_tool","arguments":{{"input_expression":"150 + 60"}}}}
    {{"tool_name":"calculator_tool","arguments":{{"input_expression":"200 + 60"}}}}



- get_activities_by_date_tool:
  - Purpose: retrieve candidate activities for a given date and city to replace or add activities.
  - Arguments: date:"<string>YYYY-MM-DD", city: <string>.

- run_evals_tool:
  - Purpose: run all evaluation functions to verify alignment with feedback, weather, and constraints.
  - Call it once you have a complete and budget-checked TravelPlan.
  - Arguments: travel_plan: {TravelPlan}.
  - If evaluations fail, further refinement steps are allowed; when you propose another evaluation, provide a new ACTION accordingly.

- final_answer_tool:
  - Purpose: return the final TravelPlan after evaluations pass.
  - Arguments: final_output: {TravelPlan}.


## Output Format (YOU MUST FOLLOW THE FORMAT)

    THOUGHT:
    Briefly describe your task, then decide what your next step 
    will be based on that information and explain your decision.

    ACTION:
    {{"tool_name": "tool_name", "arguments": {{"arg1": "value1"}}}}
    ## For the arguments, follow the instructions in the Available Tools section

## Context

The output format should include a THOUGHT and a single ACTION (tool call using JSON format).
You should not execute the tools, only analyse the task and decide which action it will take to be solved;
Use the following objects and schemas:
{TravelPlan}
{ItineraryDay}
{Weather}
{Activity}
{ActivityRecommendation}
Activities dataframe: {activities_for_dates_df}  
Weather dataframe: {weather_for_dates_df}
Tools available: {ALL_TOOLS}  
Evaluation functions: {ALL_EVAL_FUNCTIONS}
"""  # noqa


class ItineraryRevisionAgent(ChatAgent):
    system_prompt = ITINERARY_REVISION_AGENT_SYSTEM_PROMPT
    tools = ALL_TOOLS

    def get_observation_string(self, tool_call_obj) -> str:
        """Extracts the observation from the thought-action response."""

        if "tool_name" not in tool_call_obj:
            return "OBSERVATION: No tool name specified."

        if "arguments" not in tool_call_obj:
            return "OBSERVATION: No arguments specified."

        # If the arguments are not a dictionary, state the error
        if not isinstance(tool_call_obj["arguments"], dict):
            return f"OBSERVATION: Arguments should be a dictionary, got {type(tool_call_obj['arguments'])} instead."

        # If the tool name is not a string, state the error
        if not isinstance(tool_call_obj["tool_name"], str):
            return f"OBSERVATION: Tool name should be a string, got {type(tool_call_obj['tool_name'])} instead."

        tool_name = tool_call_obj["tool_name"]
        arguments = tool_call_obj["arguments"]

        tool_fn = None

        for tool in self.tools:
            if tool.__name__ == tool_name:
                tool_fn = tool
                break

        if tool_fn is None:
            return f"OBSERVATION: Unknown tool name '{tool_name}' in action string."

        if tool_name == "calculator_tool":
            print(type(arguments))
            print(arguments)

        try:
            tool_response = tool_fn(**arguments)
            return f"OBSERVATION: Tool {tool_name} called successfully with response: {tool_response}"
        except Exception as e:
            return f"OBSERVATION: Error occurred while calling tool {tool_name}: {e}"

    def run_react_cycle(
        self, original_travel_plan: TravelPlan, max_steps: int = 10, model: Optional[OpenAIModel] = None, client = None,
    ) -> TravelPlan:
        """Runs the ReAct cycle to revise the itinerary based on the evaluation results."""
        from json_repair import repair_json

        # Provide the original travel plan to revise
        self.add_message(
            role="user",
            content=f"Here is the itinerary for revision:\n{original_travel_plan.model_dump_json()}",
        )
        resp = None

        # Run the ReAct cycle for a maximum number of steps
        for step in range(max_steps):
            # Get the thought-action response from the agent
            resp = self.get_response(model=model, client=client) or ""

            # If there is no action, report it to the LLM and continue
            if "ACTION:" not in resp:
                self.add_message(role="user", content="No action found in response.")
                continue

            action_string = resp.split("ACTION:")[1].strip()

            # Parse the tool call JSON from the action string
            try:
                # Fix any JSON formatting issues. e.g. missing closing braces, etc.
                action_string = repair_json(action_string)
                tool_call_obj = json.loads(action_string)
            except json.JSONDecodeError:
                print(f"Invalid JSON in action string: {action_string}")
                self.add_message(
                    role="user",
                    content=f"Invalid JSON in action string: {action_string}",
                )
                continue

            tool_name = tool_call_obj.get("tool_name", None)

            # If the final answer tool is called, validate and return the final travel plan
            if tool_name == "final_answer_tool":
                try:
                    new_travel_plan = TravelPlan.model_validate(
                        tool_call_obj["arguments"].get("final_output", tool_call_obj["arguments"])
                    )
                    return new_travel_plan
                except Exception as e:
                    self.add_message(
                        role="user", content=f"Error validating final answer: {e}"
                    )
                    continue

            # For all other tools, execute the tool call and add the observation
            else:
                # Add the 
                observation_string = self.get_observation_string(
                    tool_call_obj=tool_call_obj
                )
                self.add_message(role="user", content=observation_string)

        raise RuntimeError(
            f"ReAct cycle did not complete within {max_steps} steps. Last response: {resp}"
        )

# Instantiate the Itinerary Revision Agent
itinerary_revision_agent = ItineraryRevisionAgent()

# Let's get a single THOUGHT/ACTION response back to check that the agent is working as expected.
resp = itinerary_revision_agent.chat(
    user_message=f"Here is the itinerary for revision: {travel_plan_1.model_dump_json(indent=2)}",
    add_to_messages=False,
    model=MODEL,
    client=client,
) or ""

print_in_box(resp, "Raw Response")
# Check for THOUGHT
if "THOUGHT:" in resp:
    print("✅ `THOUGHT:` found in raw the response, as expected.")
else:
    print("❌ Expected `THOUGHT:` in raw the response. Please check the system prompt (output format).")
# Check for ACTION
if "ACTION:" in resp:
    print("✅ `ACTION:` found in raw the response, as expected.")
else:
    print("❌ Expected `ACTION:` in raw the response. Please check the system prompt (output format).")
if "\"tool_name\"" in resp:
    print("✅ `\"tool_name\":` found in raw the response, as expected.")
else:
    print("❌ Expected `\"tool_name\":` in raw the response. Please check the system prompt (output format).")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ Your job is to analyze the current TravelPlan and decide the next tool to call                                       ║
║ (without executing it), along with the correct arguments, until the plan is ready.                                   ║
║ ## Task                                                                                                              ║
║ I want to have at least two activities per day.                                                                      ║
║ 1. Verify whether the TravelPlan satisfies all TRAVELER_FEEDBACK.                                                    ║
║ 2. Ensure every activity in the TravelPlan has a matching ID in activities_for_dates_df.                             ║
║ 3. If corrections are needed, refine the TravelPlan accordingly.                                                     ║
║ 4. The plan **must have** at 

In [ ]:
# Now let's run the ReAct cycle multiple times to get the revised itinerary.
# Note: If this takes more than a few minutes, then the agent may be stuck in a loop.
# Examine the traces to understand where it is failing and see if adjusting the system prompt helps.
# Since LLMs are stochastic, it's normal to get different results each time you run this cell.

itinerary_revision_agent = ItineraryRevisionAgent()
travel_plan_2 = itinerary_revision_agent.run_react_cycle(
    original_travel_plan=travel_plan_1, max_steps=15,
    model=MODEL,
    client=client,
)

print("✅ Revised itinerary generated successfully. Congratulations!")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ Your job is to analyze the current TravelPlan and decide the next tool to call                                       ║
║ (without executing it), along with the correct arguments, until the plan is ready.                                   ║
║ ## Task                                                                                                              ║
║ I want to have at least two activities per day.                                                                      ║
║ 1. Verify whether the TravelPlan satisfies all TRAVELER_FEEDBACK.                                                    ║
║ 2. Ensure every activity in the TravelPlan has a matching ID in activities_for_dates_df.                             ║
║ 3. If corrections are needed, refine the TravelPlan accordingly.                                                     ║
║ 4. The plan **must have** at 

In [ ]:
# double checking that the revised travel plan passes all evaluation functions.

eval_results_2 = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_2,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

assert eval_results_2.success, f"❌ Read the traces above and modify the system prompt.\n\nFailures: {eval_results_2.failures}"

print("✅ All evaluation functions passed successfully for the revised travel plan.")

eval_results_2

✅ Traveler Yuri has a match with interest {technology} at FutureTech Breakfast Meet-Up
✅ Traveler Yuri has a match with interest {technology} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art, reading} at AgentsVille Twilight Writing Escape
✅ Traveler Hiro has a match with interest {music} at Morning Groove Dance Party
✅ Traveler Hiro has a match with interest {music} at Tech Lunch & Learn: AI Frontiers
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Hiro has a match with interest {art, reading} at AgentsVille Twilight Writing Escape
✅ Traveler Anna has a match with interest {dancing} at Morning Groove Dance Party
✅ Traveler Anna has a match with interest {photography} at Tech Lunch & Learn: AI Frontiers
✅ Trave

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [ ]:
# Show the final travel plan in a readable format.

from IPython.display import display

for itinerary_day in travel_plan_2.itinerary_days:
    print(f"Date: {itinerary_day.date}")
    print(
        f"Weather: {itinerary_day.weather.condition} ({itinerary_day.weather.temperature}°{itinerary_day.weather.temperature_unit})"
    )

    activities_df = pd.DataFrame(
        [
            activity_recommendation.activity.model_dump()
            for activity_recommendation in itinerary_day.activity_recommendations
        ]
    )
    display(activities_df)

Date: 2025-06-10
Weather: clear (31.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00,2025-06-10 11:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"


Date: 2025-06-11
Weather: partly cloudy (34.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00,2025-06-11 10:30,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
1,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00,2025-06-11 13:30,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI creativity).",20,"[technology, music, photography, writing]"
2,event-2025-06-11-3,Palette & Palate: Art Meets Cooking Experience,2025-06-11 18:30,2025-06-11 20:30,"The Creative Canvas Studio, Artisanal Lane, AgentsVille","Immerse yourself in a colorful evening where art and cooking blend together! At 'Palette & Palate,' participants will begin indoors at The Creative Canvas Studio with a guided session to paint their own culinary-inspired masterpiece. Afterwards, a local chef will lead an interactive cooking class, teaching you how to craft vibrant, edible works of art. Whether you're an art enthusiast, a food lover, or both, this creative night is perfect for socializing and expressing yourself through color and flavor! All materials and ingredients are provided. This event is held indoors and welcomes all experience levels in art and cooking.",25,"[art, cooking]"


Date: 2025-06-12
Weather: thunderstorm (28.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-12-3,Tech & Film Fusion Night,2025-06-12 19:00,2025-06-12 21:30,"Virtual Reality Theater, Silicon Plaza, AgentsVille","Dive into an immersive evening where the magic of movies meets the latest in technology! Join fellow movie buffs and tech enthusiasts for a special screening of cutting-edge sci-fi short films, followed by an interactive panel with local filmmakers and VR technologists. Experience the future of entertainment and discuss how technology is transforming the world of cinema. This exciting, indoor event at the Virtual Reality Theater is perfect for anyone interested in technology and movies.",15,"[technology, movies]"
1,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"


## And, just for fun!

In [ ]:
# And finally, just for fun, let's narrate the trip.

from project_lib import narrate_my_trip

narrate_my_trip(
    vacation_info=vacation_info,
    itinerary=travel_plan_2,
    client=client,
    model=MODEL, 
)


The trip to AgentsVille includes three travelers: Yuri, age 30, Hiro, age 25, and Anna, age 26. Each traveler has distinct interests and cuisine preferences. Yuri is passionate about tennis, cooking, comedy, and technology, with a taste for Brazilian, Thai, and Mexican cuisines. Hiro enjoys reading, music, theatre, and art, favoring Indian and Japanese dishes. Anna’s interests lie in photography, sports, dancing, and movies, and she prefers Italian, Mediterranean, and Middle Eastern cuisines. The group’s budget for the trip is 130 units, and the journey spans three days from June 10 to June 12, 2025.

On June 10, the weather will be clear with a warm temperature around 31°C. The day starts with the FutureTech Breakfast Meet-Up, a perfect event for Yuri’s enthusiasm for technology. This indoor gathering offers a chance to explore the latest tech trends, network, and enjoy fresh coffee and pastries in a comfortable setting. In the evening, the AgentsVille Twilight Writing Escape provides a cozy indoor environment where Hiro can immerse himself in writing and storytelling alongside fellow enthusiasts, engaging in creative activities and socializing.

June 11 will be partly cloudy with even warmer temperatures near 34°C. The Morning Groove Dance Party kicks off the day, energizing Anna and those who enjoy dancing and music with lively, inclusive dance sessions indoors. Later, the Tech Lunch & Learn: AI Frontiers event offers a stimulating afternoon focused on artificial intelligence, appealing to Yuri’s technology interest and touching on Hiro’s music, photography, and writing passions. The day concludes with Palette & Palate: Art Meets Cooking Experience, an interactive indoor event that combines painting and cooking, engaging both Hiro’s love of art and Yuri’s cooking enthusiasm, while providing a vibrant social experience.

The final day, June 12, is expected to bring thunderstorms with cooler temperatures around 28°C. Activities are planned indoors to ensure comfort and enjoyment despite the weather. The Tech & Film Fusion Night offers a unique fusion of cinema and technology, catering to Yuri’s interest in technology and Anna’s fondness for movies, providing an immersive and engaging evening experience. To complement this, the AgentsVille Twilight Writing Escape is again available, providing Hiro another opportunity to engage in creative writing and connect with like-minded participants in a cozy indoor setting.

This itinerary thoughtfully balances the group's varied interests and ensures indoor options during warm or inclement weather, all within the planned budget. It offers a mix of technology, art, dance, cooking, and storytelling events that promise a memorable and enjoyable time in AgentsVille.